# StandardScaler

In [21]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score

In [22]:
riemann_features = pd.read_csv("../results/feature_importance_mrmr_results.csv")["feature"].tolist()
random_forest_features = pd.read_csv("../results/random_forest_feature_selection_v3.csv")["feature"].tolist()
gevrey_features = pd.read_csv("../results/gevrey_method_feature_selection_v3.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection_v3.csv")["feature"].tolist()


features_map = {
    "Random Forest (5)": random_forest_features[:5],
    "Random Forest (10)": random_forest_features[:10],
    f"Random Forest ({len(random_forest_features)})": random_forest_features,
    "Gevrey (10)": gevrey_features[:10],
    f"Gevrey ({len(gevrey_features)})": gevrey_features,
    "Correlation (10)": correlation_features[:10],
    "MRMR (10)": riemann_features[:10],
    "MRMR (50)": riemann_features[:50],
}


In [23]:
class DatasetScalerService:
    MAX_LIMIT = 100_000
    def __init__(self, features: list[str]):
        self.__scaler_X = StandardScaler()
        self.__scaler_y = StandardScaler()

        dataset = pd.read_csv("../dataset/riemann_features.csv")

        self.__X_original = dataset.drop(columns=["distance"])
        self.__y_original = dataset["distance"]
        self.__features = features

    def get_scaled_data(self, limit: int = 11_000):
        if limit > self.MAX_LIMIT:
            limit = self.MAX_LIMIT

        X = self.__X_original[self.__features].to_numpy()[:limit]
        y = self.__y_original.to_numpy()[:limit]

        split = int(0.8 * len(X))
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]

        X_train_scaled = self.__scaler_X.fit_transform(X_train)
        X_test_scaled  = self.__scaler_X.transform(X_test)
        y_train_scaled = self.__scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test_scaled  = self.__scaler_y.transform(y_test.reshape(-1, 1)).ravel()
        
        return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled

    @property
    def scaler_y(self) -> StandardScaler:
        return self.__scaler_y

In [24]:
C_values = (0.1, 1.0, 10.0)
epsilon_values = (0.01, 0.1, 1, 0.02)
best_results = []

for name, features in features_map.items():
    print(f"Evaluating feature set: {name} with {len(features)} features")

    best_rmse = np.inf
    best_r2   = -np.inf
    best_result = None
    dataset_scaler_service = DatasetScalerService(features)
    (
        X_train_scaled,
        X_test_scaled,
        y_train_scaled,
        y_test_scaled
    ) = dataset_scaler_service.get_scaled_data()
   
    scaler_y = dataset_scaler_service.scaler_y

    for C in C_values:
        for epsilon in epsilon_values:
            model = SVR(kernel="rbf", C=C, epsilon=epsilon)
            model.fit(X_train_scaled, y_train_scaled)
            y_pred_scaled = model.predict(X_test_scaled)

            y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
            y_test = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1)).ravel()

            rmse = root_mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            if rmse < best_rmse and r2 > best_r2:
                best_rmse = rmse
                best_r2 = r2
                best_result = {
                    "Group": name,
                    "model": "SVR",
                    "Best RMSE": rmse,
                    "R2": r2,
                    "Best C": C,
                    "Best Epsilon": epsilon,
                    "Features": len(features)
                }
    if best_result:
        best_results.append(best_result)

svr_df = pd.DataFrame(best_results).sort_values("Best RMSE").reset_index(drop=True)
print(svr_df)
svr_df.to_csv("../results/experiment_9.csv", index=False)

Evaluating feature set: Random Forest (5) with 5 features
Evaluating feature set: Random Forest (10) with 10 features
Evaluating feature set: Random Forest (20) with 20 features
Evaluating feature set: Gevrey (10) with 10 features
Evaluating feature set: Gevrey (24) with 24 features
Evaluating feature set: Correlation (10) with 10 features
Evaluating feature set: MRMR (10) with 10 features
Evaluating feature set: MRMR (50) with 50 features
                Group model  Best RMSE        R2  Best C  Best Epsilon  \
0  Random Forest (10)   SVR   0.036168  0.979726    10.0          0.01   
1           MRMR (10)   SVR   0.036308  0.979568    10.0          0.01   
2   Random Forest (5)   SVR   0.055782  0.951774    10.0          0.01   
3         Gevrey (10)   SVR   0.062496  0.939465    10.0          0.01   
4           MRMR (50)   SVR   0.063688  0.937134    10.0          0.01   
5  Random Forest (20)   SVR   0.064506  0.935509    10.0          0.02   
6         Gevrey (24)   SVR   0.077194